# Study 1 - Space

**Figure 1 (Combinatoric Space): Human vs Machine divergent thinking.**

Panel A - Divergent Thinking Score by Split (Bottom 10% / Middle 80% / Top 10%).

- Data: latest merged machine set (45-model midpoint) + refreshed human set (12,147).
- DAT scored with the Olson (2021) GloVe scorer (mean pairwise cosine distance x100).
- Each split sampled n=500 per group per side so the different-sized splits are comparable.
- Colorblind-safe palette: Human = purple (#5E348B), Machine = teal (#3CB7B0).
- Points: outlined, no fill, darkened 20%, alpha 0.36, 2x size.
- Significance: Welch t-test; stars + mean difference (Machine - Human) above the bracket.


In [ ]:
import csv, numpy as np, random
from scipy import stats
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
csv.field_size_limit(10**9); random.seed(42); np.random.seed(42)
def load(p,enc='utf-8-sig'):
    with open(p,newline='',encoding=enc,errors='replace') as f: return list(csv.DictReader(f))
H=load("/home/user/human_data_scored.csv"); M=load("/home/user/machine_data_merged.csv")
def sc(rows,k):
    o=[]
    for r in rows:
        try:o.append(float(r.get(k,'')))
        except:pass
    return np.array(o)
Hs=sc(H,'word_dat_score'); Ms=sc(M,'dat_score')
def splits(ss):
    lo,hi=np.percentile(ss,10),np.percentile(ss,90)
    return [ss[ss<=lo], ss[(ss>lo)&(ss<hi)], ss[ss>=hi]]
Hsp=splits(Hs); Msp=splits(Ms)
# SAMPLE: fixed n per group per side (not whole population)
SAMPLE_N=500
def samp(a): return np.random.choice(a, min(SAMPLE_N,len(a)), replace=False)
Hsp=[samp(a) for a in Hsp]; Msp=[samp(a) for a in Msp]
HUMAN="#5E348B"; MACHINE="#3CB7B0"
def darken(hexc,f=0.64):
    hexc=hexc.lstrip('#'); r,g,b=[int(hexc[i:i+2],16) for i in (0,2,4)]
    return (r*f/255,g*f/255,b*f/255)
HUMAN_D=darken(HUMAN); MACHINE_D=darken(MACHINE)
groups=["Bottom 10%","Middle 80%","Top 10%"]
def ci95(a): return 1.96*np.std(a,ddof=1)/np.sqrt(len(a))
def stars(p): return "***" if p<1e-3 else "**" if p<1e-2 else "*" if p<0.05 else "ns"
ANNOT_SIZE=10; ANNOT_COLOR="#333333"
fig,ax=plt.subplots(figsize=(7,7))
x=np.arange(3); w=0.36
for i in range(3):
    h=Hsp[i]; m=Msp[i]
    ax.bar(i-w/2, h.mean(), w, color=HUMAN, alpha=0.85, zorder=2)
    ax.bar(i+w/2, m.mean(), w, color=MACHINE, alpha=0.85, zorder=2)
    def jit(vals,center):
        return center+np.random.uniform(-w/2.6,w/2.6,len(vals)), vals
    jx,jv=jit(h,i-w/2); ax.scatter(jx,jv,s=12,facecolors='none',edgecolors=HUMAN_D,alpha=0.36,linewidths=0.6,zorder=3)
    jx,jv=jit(m,i+w/2); ax.scatter(jx,jv,s=12,facecolors='none',edgecolors=MACHINE_D,alpha=0.36,linewidths=0.6,zorder=3)
    ax.errorbar(i-w/2,h.mean(),yerr=ci95(h),color='black',capsize=4,lw=1.4,zorder=5)
    ax.errorbar(i+w/2,m.mean(),yerr=ci95(m),color='black',capsize=4,lw=1.4,zorder=5)
    t,p=stats.ttest_ind(h,m,equal_var=False)
    diff=m.mean()-h.mean()
    ytop=max(h.mean(),m.mean())+9
    ax.plot([i-w/2,i-w/2,i+w/2,i+w/2],[ytop-1.5,ytop,ytop,ytop-1.5],color='black',lw=1.1,zorder=5)
    ax.text(i,ytop+0.3,stars(p),ha='center',va='bottom',fontsize=ANNOT_SIZE,color=ANNOT_COLOR,zorder=6)
    ax.text(i,ytop+3.0,f"{diff:+.1f}",ha='center',va='bottom',fontsize=ANNOT_SIZE,weight='bold',color=ANNOT_COLOR,zorder=6)
    ax.text(i-w/2,ytop-3.2,f"{h.mean():.1f}",ha='center',va='top',fontsize=ANNOT_SIZE,weight='bold',color=ANNOT_COLOR,zorder=6)
    ax.text(i+w/2,ytop-3.2,f"{m.mean():.1f}",ha='center',va='top',fontsize=ANNOT_SIZE,weight='bold',color=ANNOT_COLOR,zorder=6)
ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=11)
ax.set_ylabel("Divergent thinking score", fontsize=11); ax.set_ylim(60,105)
from matplotlib.patches import Patch
leg=ax.legend(handles=[Patch(color=HUMAN,label='Human'),Patch(color=MACHINE,label='Machine')], loc='upper left', fontsize=10)
leg._legend_box.align='left'
leg.set_title(None)
# sample-size note directly under the legend box, left-aligned with it
fig.canvas.draw()
bb=leg.get_window_extent().transformed(ax.transAxes.inverted())
ax.text(bb.x0+0.008, bb.y0-0.02, f'n={SAMPLE_N} per group,\nrandom sampled', transform=ax.transAxes, fontsize=8.5, color='#555', va='top', ha='left')
ax.set_title("Panel A - Divergent Thinking Score by Split", fontsize=12, weight='bold')

ax.spines[['top','right']].set_visible(False)
# exact 1:1 aspect per Dawei's snippet
x0,x1=ax.get_xlim(); y0,y1=ax.get_ylim()
ax.set_aspect(abs(x1-x0)/abs(y1-y0))
fig.tight_layout(); fig.savefig("/home/user/panelA.png",dpi=160,bbox_inches='tight'); plt.close(fig)
print("PANEL A v4 (sampled + 1:1) DONE")
